# RO47019: Intelligent Control Systems Practical Assignment
* Period: 2025-2026, Q3
* Course homepage: https://brightspace.tudelft.nl/d2l/home/775422
* Instructor: Cosimo Della Santina (C.DellaSantina@tudelft.nl)
* Teaching assistant: Mukil Saravanan (m.saravanan-1@student.tudelft.nl)
* (c) TU Delft, 2026

Make sure you fill in any place that says `YOUR CODE HERE` or `YOUR ANSWER HERE` and remove `raise NotImplementedError()` afterwards. Moreover, if you see an empty cell, please **do not** delete it, instead run that cell as you would run all other cells. Finally, please **do not** add any extra cells to this notebook or change the existing cells unless you are explicitly asked to do so.

Please fill in your name(s) and other required details below:

In [1]:
# Please fill in your names, student numbers, netID, and emails below.
STUDENT_1_NAME = "Andrei Simionescu"
STUDENT_1_STUDENT_NUMBER = "5459559"
STUDENT_1_NETID = "asimionescu"
STUDENT_1_EMAIL = "asimionescu@tudelft.nl"

In [2]:
# Note: this block is a check that you have filled in the above information.
# It will throw an AssertionError until all fields are filled
assert STUDENT_1_NAME != ""
assert STUDENT_1_STUDENT_NUMBER != ""
assert STUDENT_1_NETID != ""
assert STUDENT_1_EMAIL != ""

### General announcements

* Do *not* share your solutions (also after the course is finished), and do *not* copy solutions from others. By submitting your solutions, you claim that you alone are responsible for this code.

* Please post your questions regarding this assignment in the correct support forum on Brightspace, this way everybody can benefit from the response. Please note that it is **not** allowed to post any code relating to solution attempts. If you do have a particular question that you want to ask directly, please use the scheduled Q&A hours to ask the TA or if not possible otherwise, send an email to the instructor or TA.

* This notebook will have in various places a line that throws a `NotImplementedError` exception. These are locations where the assignment requires you to adapt the code! These lines are just there as a reminder for you that you have not yet adapted that particular piece of code, especially when you execute all the cells. Once your solution code replaced these lines, it should accordingly *not* throw any exceptions anymore.

* This [Jupyter notebook](https://jupyter.org/) uses `nbgrader` to help us with automated tests. `nbgrader` will make various cells in this notebook "uneditable" or "unremovable" and gives them a special id in the cell metadata. This way, when we run our checks, the system will check the existence of the cell ids and verify the number of points and which checks must be run. While there are ways that you can edit the metadata and work around the restrictions to delete or modify these special cells, you should not do that since then our nbgrader backend will not be able to parse your notebook and give you points for the assignment. 

* Please note that the above mentioned _read-only_ protection only works in Jupyter Notebook, and it does not work if you open this notebook in another editor (e.g., VSCode, PyCharm, etc.). Therefore, we recommend that you only use Jupyter Notebook for this course. If you use any other editor, you may accidentally delete cells, modify the tests, etc., which would cause you to lose points.

* If you edit a function that is imported in another notebook, you need to **restart the kernel** of the notebook where you are using the function. Otherwise, the changes will not be effective.

* **IMPORTANT**: Please make sure that your code executes without any errors before submitting the notebook. An easy way to ensure this is to use the validation script as described in the README.

# Task 3h - Open questions (13p)
**Authors:** Lorenzo Lyons, Mariano Ramirez Montero(m.ramirezmontero-1@tudelft.nl)

In this last part of Problem 3, we will try to put together the insights we have learned from the previous tasks and answer some questions on how GPs can be used in Robotics. 

*Note:* To answer these questions, you are encouraged to go back to the previous tasks to put your theories to the test by temporarily modifying the code. (The final version should still answer the questions of the previous questions) 

## Task 3h-1 Behavioural cloning torques with also angular velocities (3p)

In task 3f we only used the configuration of the robot as an input to the GP model. By adding the variance minimization term, we obtained a good performance when starting from the same initial condition as the training dataset. 
How would the system behave if the initial condition was perturbed slightly (Feel free to try this out in the code)? 
Would it be possible to also provide the angular velocity as an input to the GP model? How do you expect the behavior of the controller to change?

_If the initial condition was perturbed slightly_:

With the variance minimization term (as used in 3f.6), the system is able to tolerate small perturbations. The GP uncertainty is low close to the reference trajectory (where the training data was collected from), and grows as it move away from it. Under small perturbation of the initial conditions, the robot will start in a higher uncertainty region, but the variance minimization term will push the robot towards lower uncertainty areas, effectively correcting for the unwanted behaviour. Without this term, any small perturbation will lead to divergence as the controller has no mechanism to return back to the known trajectory. GPs nicely encapsulate model uncertainty, and with the use of the variance minimization term, it is able to steer the robot back to low uncertainty regions (sampled reference trajectory). 

_If we provide the angular velocity as input to the GP model_:

I argued in 3f.5 that the same configuration $\theta$ can map to different torques depending on the angular velocity $\dot\theta$, and, as a result, the GP can only approximate the position part of the PD controller. Now, let's consider the input $[\theta, \dot\theta]$ instead of just $\theta$ for the GP model. This would allow the GP to approximate the full PD controller, including the derivative (damping) term. I would expect, that such a model, can achieve better tracking performance because it can now distinguish the necessary feedback torques, resolving the above mentioned ambiguity. The downside is that now we are dealing with a higher dimensional input to the GP, which will require (possibly) more training data and a higher number of inducing points (I assume it also takes a bit longer to run). 

## Task 3h-2 Reflections on variance minimization (5p)

In tasks 3f and 3g, we noticed how adding a variance minimization term significantly increased the overall performance of the controller. 

What is the meaning of the negative variance gradient? Why did the relatively simple additional variance minimization terms work well for the robotic arm? Would this strategy apply to any dynamical system, such as an autonomous vehicle?

_Meaning of the negative variance gradient_:

The variance of the GP at a given input is a measure of the epistemic uncertainty (lack of knowledge, or incomplete information). The gradient of the variance points in the direction of increasing uncertainty, and conversely, the negative of the gradient points in the direction of maximum decrease in uncertainty. If we follow this negative gradient we end up in regions of space of low uncertainty i.e regions of input space that are covered by the training data. Since the training data was collected around a reference trajectory, these coincide with regions of space that are low in epistemic uncertainty for the GP. Applying a torque proportional to this negative gradient effectively moves the robot arm closer to the reference trajectory. This implicitly steers the robot arm back to the desired path, whenever it deviates from it.

_Why it worked well for the robot arm_:

Since the training data was collected only around the ellipsoid trajectory (and nowhere else), this induces a GP which has low uncertainty around that specific path, and high everywhere else. We can think of this like an ellipsoid shaped valley, where going down a valley coincides with getting back on the desired trajectory. The force that pulls the robot down this valley is the variance minimization term. This process worked well for this simple scenario because the training data distribution was perfectly aligned with the desired behaviour. If we had wanted the robot arm to perform a different trajectory we would have had to collect new data for it, and train the GP again to match this new desired trajectory.

_Would this work for an autonomous vehicle_:

I think in general, no, but it depends on what we define as an autonomous vehicle and what task does the vehicle have to solve. If we take the example of the autonomous racing car from the recorded lecture (GPs for MPC, ETH) where the desired trajectory does not change across runs, and it's the *only path* the car has to follow, then, GPs with variance minimization terms definetely work. As explained above, the variance minimization term will steer the car back on path, and every successive run will collect more data and optimize the car's behaviour further. Instead, if we try to apply this technique to any dynamical system, say an autonomous Tesla, this will not generally work out fine. In this case, the training data will span many road scenarios, different speeds or maneuvers. The GP's low uncertainty will spread across a broad range of state space configurations and would *not correspond to only one path*. The negative variance gradient will then not steer the vehicle towards a particular trajectory, but will most likely send it down the nearest valley (on the epistemic uncertainty manifold). This strategy would fail to be an effective controller.

## Task 3h-3 Safety properties comparison (5p)

Imagine the robot is performing the same trajectory following the task as we have seen in the previous tasks when an unfortunate human happens to collide with the robot. Imagine the particular safety-critical scenario where the human is unable to remove himself/herself/themselves from the collision, i.e., the human is stuck against the robot. 

What behavior do you expect from the PD + gravity compensation, the torque cloning in configuration space, and the reference trajectory behavioral cloning? 

If the human instead managed to push the robot away from its intended trajectory, what would be the different behavior among the three controllers? Which one would be the safest?


_Scenario 1: Human stuck against the robot_:

- PD + gravity compensation:
  The controller continously computes a torque proportional to the position and velocity errors, without accounting for the mass of the human stuck to itself. If the human is blockign the robot from reaching the desired trajectory, the controllor will continue exerting the full torque against the human's body. This is the most dangerous outcome, as the robot now behaves as a stiff spring compressed against the human, which creates a force that grows with the distance from the desired position.
- Torque cloning in configuration space (3f):
  The GPs output force that is dependent on the current configuration, and does not track a reference trajectory per se. If the human is stuck to the robot, and has pushed it outside of its desired trajectory, the GP will return a torque that it has learned for that trajectory (still likely non-zero, since the robot was pushed out of it trajectory). The variance minimization term will also create a torque that pushes the robot towards the training distribution. This results in a persistent force against the human, similar to the PD, but probably of much less intensity since the GP is an approximation. If the robot was pushed further from its intended trajectory, the PD would create a much higher torque based on the position difference, than the GP will create. That is because the variance minimization term saturates in higher uncertainty regions (after a threshold, all forces are the same i.e end up in an area of very similar uncertainty, while for PD this grows proportional to the distance). 
- Reference trajectory behavioural cloning (3g):
  Here, the GP predicts a desired delta angle, and a proportional gain converts it to torque. If the human is stuck to the robot and holds it in place, the torque is determined by the GP's prediction for that configuration, and not by an explicit trajectory error (as it was in the PD case). The damping term applies torque against the current velocity, but if the robot is not moving than this torque will be 0. I think this configuration would be the safest, as the total torque applied is probably less than in the previous example, but nonetheless none are trully safe for the human.

_Scenario 2: Human pushes away the robot_:

- PD + gravity compensation:
  When the robot is let go, it will snap back aggressively to the reference trajectory with potentially high velocities. The controller is resistant to perturbations (which in this case happen to be the human), and will try to compensate with higher torques (implicitly higher accelerations) to move back to its desired position. Because of this snap back, I believe that this controller is the most dangerous for the human involved.
- Torque cloning in configuration space (3f):
  When released, the controller will apply the torque learned for the current configuration, and the variance minimizatin term will guide the robot back on the training distribution. Since there is no explcit error signal, the GP will predict a different torque than the PD controller, and the recovery will definetely be much less stiff. This scenario would be moderately safe for the human involved.
- Reference trajectory behavioural cloning (3g):
  When released, the GP will predict movement direction from the current configuration, and the controller will convert that into torque. Just as before, the variance minimization term will steer it back to the reference trajectory, but this time, the Kd damping term will reduce the velocity during recovery. This produces the a soft and gentle return back to the desired path. Thus, this controller is the safest among all three, becuase even at large distances from the reference trajectory, it will output a soft and compliant behaviour.